<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install datasets

In [2]:
from huggingface_hub import login

login("")

In [3]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4667936
    })
})

Columns:
['text']

Sample:
{'text': 'সাময়িক পত্রে প্রকাশিত'}


In [4]:
print("Rows:", len(ds["train"]))

Rows: 4667936


In [5]:
for i in range(5):
    print("\n--- SAMPLE", i, "---")
    print(ds["train"][i])


--- SAMPLE 0 ---
{'text': 'সাময়িক পত্রে প্রকাশিত'}

--- SAMPLE 1 ---
{'text': 'ও'}

--- SAMPLE 2 ---
{'text': 'পুস্তকাকারে অপ্রকাশিত রচনা'}

--- SAMPLE 3 ---
{'text': ''}

--- SAMPLE 4 ---
{'text': ''}


In [6]:
ds["train"][0]

{'text': 'সাময়িক পত্রে প্রকাশিত'}

In [7]:
ds["train"].column_names

['text']

In [8]:
print("Rows:", len(ds["train"]))

Rows: 4667936


In [9]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

Saved documents: 4604919


In [10]:
import os

size_mb = os.path.getsize("bn_literature.txt") / (1024**2)

print("Size (MB):", round(size_mb, 2))

Size (MB): 1574.18


Sampling Out

In [11]:
import random

random.seed(42)

TARGET = 250000

with open("bn_literature.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_literature.txt", encoding="utf-8") as inp, \
     open("bn_literature_250k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:

            out.write(line)
            count += 1

print("Saved:", count)

Total docs: 4604919
Saved: 250000


In [12]:
import os

print(
    "Size MB:",
    round(
        os.path.getsize(
            "bn_literature_250k.txt"
        )/(1024**2),
        2
    )
)

Size MB: 85.7


In [13]:
vocab = set()

chars = 0
words = 0

with open("bn_literature_250k.txt", encoding="utf-8") as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))

print("TTR :", len(vocab)/words)

print(
    "Average Word Length:",
    chars / words
)

Characters : 34204085
Words      : 5601171
Vocabulary : 522341
TTR : 0.09325567814301688
Average Word Length: 6.106595388714253


In [14]:
import random

random.seed(42)

train_count = 0
test_count = 0

with open("bn_literature_250k.txt", encoding="utf-8") as inp, \
     open("train_lit_bn.txt", "w", encoding="utf-8") as train_f, \
     open("test_lit_bn.txt", "w", encoding="utf-8") as test_f:

    for line in inp:

        if random.random() < 0.1:

            test_f.write(line)
            test_count += 1

        else:

            train_f.write(line)
            train_count += 1

print("Train:", train_count)
print("Test :", test_count)

Train: 224957
Test : 25043


In [15]:
import os

print(
    "Train MB:",
    round(
        os.path.getsize("train_lit_bn.txt")/(1024**2),
        2
    )
)

print(
    "Test MB:",
    round(
        os.path.getsize("test_lit_bn.txt")/(1024**2),
        2
    )
)

Train MB: 77.03
Test MB: 8.67


Transliteration

In [16]:
!pip install aksharamukha

In [19]:
from aksharamukha import transliterate
from tqdm import tqdm
import time

start = time.time()

with open("train_lit_bn.txt", encoding="utf-8") as inp, \
     open("train_lit_iso.txt", "w", encoding="utf-8") as out:

    count = 0

    for line in inp:

        line = line.strip()

        if not line:
            continue

        ascii = transliterate.process(
            "Bengali",
            "HK",
            line
        )

        out.write(ascii + "\n")

        count += 1

        if count % 10000 == 0:

            elapsed = time.time() - start

            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

10,000 docs | 665.69 docs/sec
20,000 docs | 703.59 docs/sec
30,000 docs | 713.59 docs/sec
40,000 docs | 725.33 docs/sec
50,000 docs | 727.31 docs/sec
60,000 docs | 734.89 docs/sec
70,000 docs | 729.18 docs/sec
80,000 docs | 729.28 docs/sec
90,000 docs | 704.84 docs/sec
100,000 docs | 708.05 docs/sec
110,000 docs | 710.14 docs/sec
120,000 docs | 720.97 docs/sec
130,000 docs | 730.60 docs/sec
140,000 docs | 735.08 docs/sec
150,000 docs | 736.47 docs/sec
160,000 docs | 738.52 docs/sec
170,000 docs | 740.24 docs/sec
180,000 docs | 740.53 docs/sec
190,000 docs | 744.72 docs/sec
200,000 docs | 748.94 docs/sec
210,000 docs | 749.42 docs/sec
220,000 docs | 750.22 docs/sec
Completed: 224957


In [21]:
from aksharamukha import transliterate
import time

start = time.time()

with open("test_lit_bn.txt", encoding="utf-8") as inp, \
     open("test_lit_iso.txt", "w", encoding="utf-8") as out:

    count = 0

    for line in inp:

        line = line.strip()

        if not line:
            continue

        ascii = transliterate.process(
            "Bengali",
            "HK",
            line
        )

        out.write(ascii + "\n")

        count += 1

        if count % 10000 == 0:

            elapsed = time.time() - start

            print(
                f"{count:,} docs | "
                f"{count/elapsed:.2f} docs/sec"
            )

print("Completed:", count)

10,000 docs | 745.66 docs/sec
20,000 docs | 785.48 docs/sec
Completed: 25043


In [22]:
import os

files = [
    "train_lit_bn.txt",
    "train_lit_iso.txt",
    "test_lit_bn.txt",
    "test_lit_iso.txt"
]

for f in files:

    print(
        f,
        round(
            os.path.getsize(f)/(1024**2),
            2
        ),
        "MB"
    )

train_lit_bn.txt 77.03 MB
train_lit_iso.txt 34.16 MB
test_lit_bn.txt 8.67 MB
test_lit_iso.txt 3.85 MB


In [23]:
def corpus_stats(path):

    vocab = set()

    chars = 0
    words = 0

    with open(path, encoding="utf-8") as f:

        for line in f:

            chars += len(line)

            toks = line.split()

            words += len(toks)

            vocab.update(toks)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats("train_lit_bn.txt")
ascii = corpus_stats("train_lit_iso.txt")

print("Bengali:", bn)
print("ASCII:", ascii)

print(
    "\nExpansion Factor:",
    ascii["chars"] / bn["chars"]
)

print(
    "Vocabulary Ratio:",
    ascii["vocab"] / bn["vocab"]
)

Bengali: {'chars': 30743717, 'words': 5033550, 'vocab': 486051, 'avg_word_len': 6.107760328197793}
ASCII: {'chars': 35575568, 'words': 5033546, 'vocab': 483843, 'avg_word_len': 7.067695020568006}

Expansion Factor: 1.1571654787220427
Vocabulary Ratio: 0.995457266830024


In [26]:
bn_chars = set()

with open("train_lit_bn.txt", encoding="utf-8") as f:

    for line in f:
        bn_chars.update(line)

bn_chars.discard(" ")
bn_chars.discard("\n")

missing = []

for ch in bn_chars:

    ascii = transliterate.process(
        "Bengali",
        "HK",
        ch
    )

    if not ascii:
        missing.append(ch)

print(
    "Letter Coverage:",
    1 - len(missing)/len(bn_chars)
)

print("Missing:", missing[:20])

Letter Coverage: 0.9916666666666667
Missing: ['\u200d', '\u200c', '·', '্']


In [27]:
import random

random.seed(42)

sample_lines = []

with open("test_lit_bn.txt", encoding="utf-8") as f:
    for line in f:
        sample_lines.append(line.strip())

sample_lines = random.sample(
    sample_lines,
    1000
)

print(len(sample_lines))

1000


In [29]:
from aksharamukha import transliterate

refs = []
hyps = []

for text in sample_lines:

    ascii = transliterate.process(
        "Bengali",
        "HK",
        text
    )

    back = transliterate.process(
        "HK",
        "Bengali",
        ascii
    )

    refs.append(text)
    hyps.append(back)

print("Done")

Done


In [31]:
!pip -q install jiwer

from jiwer import cer

score = cer(
    refs,
    hyps
)

print("CER:", score)

CER: 0.05352208512086034


In [32]:
from jiwer import wer

score = wer(
    refs,
    hyps
)

print("WER:", score)

WER: 0.14929324209579942


In [33]:
!pip -q install sacrebleu

import sacrebleu

bleu = sacrebleu.corpus_bleu(
    hyps,
    [refs]
)

print("BLEU:", bleu.score)

BLEU: 69.60394975575505


In [34]:
import random
from aksharamukha import transliterate

random.seed(42)

samples = []

with open("test_lit_bn.txt", encoding="utf-8") as f:
    lines = f.readlines()

samples = random.sample(lines, 20)

for i, text in enumerate(samples, 1):

    text = text.strip()

    ascii = transliterate.process(
        "Bengali",
        "HK",
        text
    )

    print(f"\n=== Sample {i} ===")
    print("BN :", text)
    print("ASCII:", ascii)


=== Sample 1 ===
BN : হেসে উঠল ব্রায়ান। খুব সহজভাবে বলল,আমার জন্ম-মৃত্যু সবই তো তোমার জন্য তোমাকে বাঁচিয়ে রাখার জন্য। বলে পিকুর দিকে তাকাল। ব্রায়ানের গভীর দৃষ্টির দিকে তাকিয়ে পিকু বুঝল যে ও কথাটা হালকাভাবে বলছে না।
ASCII: hese uThala brAYAna. khuba sahajabhAbe balala,AmAra janma-mRtyu saba_i to tomAra janya tomAke bA~ciYe rAkhAra janya. bale pikura dike tAkAla. brAYAnera gabhIra dRSTira dike tAkiYe piku bujhala ye o kathATA hAlakAbhAbe balache nA.

=== Sample 2 ===
BN : ‘ধন্যবাদ সরদার।’ বলে প্রফেসর আরাপাহো দুহাত তুলে কমিউনিটির সব নেতাদের দিকে তাকিয়ে বলল, ‘সকলকে গুড ইভিনিং।’
ASCII: ‘dhanyabAda saradAra.’ bale praphesara ArApAho duhAta tule kamiuniTira saba netAdera dike tAkiYe balala, ‘sakalake guDa ibhiniM.’

=== Sample 3 ===
BN : কেন কাঁদি বুঝিতে পার না?
ASCII: kena kA~di bujhite pAra nA?

=== Sample 4 ===
BN : কথাটা শুনে ঘাড়ের কাছে শিরশির করে উঠল রানার।
ASCII: kathATA zune ghAr3era kAche zirazira kare uThala rAnAra.

=== Sample 5 ===
BN : সুমিত্রা তবুও চুপ করে রইল।
ASCII: sumi

In [36]:
import random
from aksharamukha import transliterate

random.seed(42)

with open("test_lit_bn.txt", encoding="utf-8") as f:
    lines = f.readlines()

for text in random.sample(lines, 20):

    text = text.strip()

    ascii = transliterate.process(
        "Bengali",
        "HK",
        text
    )

    back = transliterate.process(
        "HK",
        "Bengali",
        ascii
    )

    print("\n--------------------")
    print("Original :", text)
    print("ASCII      :", ascii)
    print("Back BN  :", back)
    print("Match    :", text == back)


--------------------
Original : হেসে উঠল ব্রায়ান। খুব সহজভাবে বলল,আমার জন্ম-মৃত্যু সবই তো তোমার জন্য তোমাকে বাঁচিয়ে রাখার জন্য। বলে পিকুর দিকে তাকাল। ব্রায়ানের গভীর দৃষ্টির দিকে তাকিয়ে পিকু বুঝল যে ও কথাটা হালকাভাবে বলছে না।
ASCII      : hese uThala brAYAna. khuba sahajabhAbe balala,AmAra janma-mRtyu saba_i to tomAra janya tomAke bA~ciYe rAkhAra janya. bale pikura dike tAkAla. brAYAnera gabhIra dRSTira dike tAkiYe piku bujhala ye o kathATA hAlakAbhAbe balache nA.
Back BN  : হেসে উঠল ব্রায়ান। খুব সহজভাবে বলল,আমার জন্ম-মৃত্যু সবই তো তোমার জন্য তোমাকে বাঁচিয়ে রাখার জন্য। বলে পিকুর দিকে তাকাল। ব্রায়ানের গভীর দৃষ্টির দিকে তাকিয়ে পিকু বুঝল যে ও কথাটা হালকাভাবে বলছে না।
Match    : False

--------------------
Original : ‘ধন্যবাদ সরদার।’ বলে প্রফেসর আরাপাহো দুহাত তুলে কমিউনিটির সব নেতাদের দিকে তাকিয়ে বলল, ‘সকলকে গুড ইভিনিং।’
ASCII      : ‘dhanyabAda saradAra.’ bale praphesara ArApAho duhAta tule kamiuniTira saba netAdera dike tAkiYe balala, ‘sakalake guDa ibhiniM.’
Back BN  : ‘ধন্যবাদ সরদ

In [38]:
print(repr(text))
print(repr(back))

'টুনি বলল, আ লো আমার সুনা লো, কিচ্ছু বুঝি হয় না তর? পাখি গ্যাছে গা শুইনা তো বুকখানে ঢেকির পাড় পড়তে আছিল। বলেই গলা চড়িয়ে ডাকতে লাগল, পাখি-পাখি—পাখি–'
'টুনি বলল, আ লো আমার সুনা লো, কিচ্ছু বুঝি হয় না তর? পাখি গ্যাছে গা শুইনা তো বুকখানে ঢেকির পাড় পড়তে আছিল। বলেই গলা চড়িয়ে ডাকতে লাগল, পাখি-পাখি—পাখি–'


In [39]:
for i, (a, b) in enumerate(zip(text, back)):
    if a != b:
        print(i, repr(a), ord(a))
        print(i, repr(b), ord(b))
        break

42 'য' 2479
42 'য়' 2527


Tokenization

In [40]:
chars = set()

with open("train_lit_bn.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

lit_bn_chars = sorted(list(chars))

print("Character inventory:", len(lit_bn_chars))
print(lit_bn_chars[:50])

Character inventory: 480
['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R']


In [41]:
chars = set()

with open("train_lit_iso.txt", encoding="utf-8") as f:
    for line in f:
        chars.update(line)

chars.discard(" ")
chars.discard("\n")

lit_iso_chars = sorted(list(chars))

print("Character inventory:", len(lit_iso_chars))
print(lit_iso_chars[:50])

Character inventory: 400
['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R']


In [42]:
print(train_count)
print(test_count)

224957
25043


In [43]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_lit_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_lit_bn_{vocab_size}.json"
    )

print("Finished.")


Training BPE-5000

Training BPE-10000

Training BPE-15000

Training BPE-20000

Training BPE-25000

Training BPE-30000

Training BPE-35000

Training BPE-40000

Training BPE-45000

Training BPE-50000
Finished.


In [44]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_path
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                enc = tokenizer.encode(word)

                toks = enc.tokens

                total_tokens += len(toks)

                sent_tokens += len(toks)

                total_chars += len(word)

                if len(toks) > 1:
                    fragmented += 1

                oov += toks.count("[UNK]")

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [45]:
results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_tokenizer(
        f"bpe_lit_bn_{vocab_size}.json",
        "test_lit_bn.txt",
        vocab_size
    )

    print(res)

    results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 2.642608360155808e-05, 'fertility': 1.731817533177948, 'cpt': 2.9427262628482143, 'compression': 2.9427262628482143, 'avg_tokens': np.float64(39.25312462564389), 'wfr': 0.4545656344638412, 'variance': np.float64(3877.0655449825513)}
{'vocab': 10000, 'oov': 2.642608360155808e-05, 'fertility': 1.561318203519602, 'cpt': 3.264078344731714, 'compression': 3.264078344731714, 'avg_tokens': np.float64(35.38861158806852), 'wfr': 0.3796723517981188, 'variance': np.float64(3073.3882534850027)}
{'vocab': 15000, 'oov': 2.642608360155808e-05, 'fertility': 1.487251176401155, 'cpt': 3.42663365691576, 'compression': 3.42663365691576, 'avg_tokens': np.float64(33.70981911112886), 'wfr': 0.34493438403441734, 'variance': np.float64(2761.124715708205)}
{'vocab': 20000, 'oov': 2.642608360155808e-05, 'fertility': 1.4429469663736896, 'cpt': 3.5318449368595455, 'compression': 3.5318449368595455, 'avg_tokens': np.float64(32.70562632272491), 'wfr': 0.32225199560974666, 'variance': np.float6

In [46]:
import csv

with open(
    "literature_bpe_bn_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

    for r in results:

        writer.writerow([
            "Bengali",
            "BPE",
            r["vocab"],
            r["oov"],
            r["fertility"],
            r["cpt"],
            r["compression"],
            r["avg_tokens"],
            r["wfr"],
            r["variance"]
        ])

print(
    "Saved literature_bpe_bn_results.csv"
)

Saved literature_bpe_bn_results.csv


In [47]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_lit_bn_5000.json"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )


 বাংলাদেশ
['বাংলা', 'দেশ']

 কলকাতা
['কলকাতা']

 সাহিত্য
['সাহিত্য']

 কবিতা
['কবিতা']

 উপন্যাস
['উপন্যা', 'স']


In [48]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO BPE-{vocab_size}")

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_lit_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_lit_iso_{vocab_size}.json"
    )

print("Finished.")


Training ISO BPE-5000

Training ISO BPE-10000

Training ISO BPE-15000

Training ISO BPE-20000

Training ISO BPE-25000

Training ISO BPE-30000

Training ISO BPE-35000

Training ISO BPE-40000

Training ISO BPE-45000

Training ISO BPE-50000
Finished.


In [49]:
import numpy as np
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file,
    vocab_size
):

    tokenizer = Tokenizer.from_file(
        tokenizer_path
    )

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                enc = tokenizer.encode(word)

                toks = enc.tokens

                total_tokens += len(toks)

                sent_tokens += len(toks)

                total_chars += len(word)

                if len(toks) > 1:
                    fragmented += 1

                oov += toks.count("[UNK]")

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [50]:
iso_bpe_results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_tokenizer(
        f"bpe_lit_iso_{vocab_size}.json",
        "test_lit_iso.txt",
        vocab_size
    )

    print(res)

    iso_bpe_results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 2.6426130157499735e-05, 'fertility': 1.7387142806807372, 'cpt': 3.4842536798493104, 'compression': 3.4842536798493104, 'avg_tokens': np.float64(39.40937587349758), 'wfr': 0.45701878016983194, 'variance': np.float64(3922.4949518246576)}
{'vocab': 10000, 'oov': 2.6426130157499735e-05, 'fertility': 1.5770991156055107, 'cpt': 3.8413068466799896, 'compression': 3.8413068466799896, 'avg_tokens': np.float64(35.74623647326598), 'wfr': 0.38593953701419964, 'variance': np.float64(3132.9467049789405)}
{'vocab': 15000, 'oov': 2.6426130157499735e-05, 'fertility': 1.5068936964870865, 'cpt': 4.020271399894545, 'compression': 4.020271399894545, 'avg_tokens': np.float64(34.15497344567344), 'wfr': 0.352124660864663, 'variance': np.float64(2829.990478299618)}
{'vocab': 20000, 'oov': 2.6426130157499735e-05, 'fertility': 1.4649906627673444, 'cpt': 4.135262964218972, 'compression': 4.135262964218972, 'avg_tokens': np.float64(33.20520704388452), 'wfr': 0.3304199992953032, 'variance': n

In [51]:
import csv

with open(
    "literature_bpe_iso_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

    for r in iso_bpe_results:

        writer.writerow([
            "ISO",
            "BPE",
            r["vocab"],
            r["oov"],
            r["fertility"],
            r["cpt"],
            r["compression"],
            r["avg_tokens"],
            r["wfr"],
            r["variance"]
        ])

print("Saved.")

Saved.


In [52]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "bpe_lit_iso_5000.json"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)

    print(
        tok.encode(w).tokens
    )


 bāṃlā
['b', 'ā', 'ṃ', 'l', 'ā']

 kalakātā
['kala', 'k', 'ā', 't', 'ā']

 sāhitya
['s', 'ā', 'hi', 'tya']

 kabitā
['kabi', 't', 'ā']

 upanyāsa
['upa', 'n', 'y', 'ā', 'sa']


In [53]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Bengali WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_lit_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_lit_bn_{vocab_size}.json"
    )

print("Finished.")


Training Bengali WordPiece-5000

Training Bengali WordPiece-10000

Training Bengali WordPiece-15000

Training Bengali WordPiece-20000

Training Bengali WordPiece-25000

Training Bengali WordPiece-30000

Training Bengali WordPiece-35000

Training Bengali WordPiece-40000

Training Bengali WordPiece-45000

Training Bengali WordPiece-50000
Finished.


In [54]:
wp_bn_results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_tokenizer(
        f"wp_lit_bn_{vocab_size}.json",
        "test_lit_bn.txt",
        vocab_size
    )

    print(res)

    wp_bn_results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.9379127974475928e-05, 'fertility': 1.8236851702104044, 'cpt': 2.794487239678427, 'compression': 2.794487239678427, 'avg_tokens': np.float64(41.335383140997486), 'wfr': 0.48316746561526086, 'variance': np.float64(4335.68830080257)}
{'vocab': 10000, 'oov': 1.9379127974475928e-05, 'fertility': 1.6165311008577907, 'cpt': 3.1525931883645733, 'compression': 3.1525931883645733, 'avg_tokens': np.float64(36.64005909835084), 'wfr': 0.3981706103192095, 'variance': np.float64(3323.542846013359)}
{'vocab': 15000, 'oov': 1.9379127974475928e-05, 'fertility': 1.5280371938317998, 'cpt': 3.335170739243624, 'compression': 3.335170739243624, 'avg_tokens': np.float64(34.63426905722158), 'wfr': 0.3593119352525717, 'variance': np.float64(2932.782385109416)}
{'vocab': 20000, 'oov': 1.9379127974475928e-05, 'fertility': 1.476555659498151, 'cpt': 3.4514546773508457, 'compression': 3.4514546773508457, 'avg_tokens': np.float64(33.46739607874456), 'wfr': 0.33522544091920486, 'variance': np.

## Table 1. BPE Results (Bangla)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |    Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | ----------: |
|  5,000 | 0.000026 |     1.7318 |     2.9427 |     39.2531 |     0.4546 |     3877.07 |
| 10,000 | 0.000026 |     1.5613 |     3.2641 |     35.3886 |     0.3797 |     3073.39 |
| 15,000 | 0.000026 |     1.4873 |     3.4266 |     33.7098 |     0.3449 |     2761.12 |
| 20,000 | 0.000026 |     1.4429 |     3.5318 |     32.7056 |     0.3223 |     2587.08 |
| 25,000 | 0.000026 |     1.4137 |     3.6048 |     32.0437 |     0.3068 |     2475.81 |
| 30,000 | 0.000026 |     1.3930 |     3.6585 |     31.5732 |     0.2958 |     2395.09 |
| 35,000 | 0.000026 |     1.3766 |     3.7020 |     31.2022 |     0.2870 |     2332.70 |
| 40,000 | 0.000026 |     1.3641 |     3.7361 |     30.9174 |     0.2801 |     2289.84 |
| 45,000 | 0.000026 |     1.3538 |     3.7644 |     30.6854 |     0.2744 |     2252.43 |
| 50,000 | 0.000026 | **1.3454** | **3.7880** | **30.4942** | **0.2698** | **2224.00** |

---

## Table 2. BPE Results (ASCII)

|  Vocab |      OOV |  Fertility |        CPT | Avg. Tokens |        WFR |    Variance |
| -----: | -------: | ---------: | ---------: | ----------: | ---------: | ----------: |
|  5,000 | 0.000026 |     1.7387 |     3.4843 |     39.4094 |     0.4570 |     3922.49 |
| 10,000 | 0.000026 |     1.5771 |     3.8413 |     35.7462 |     0.3859 |     3132.95 |
| 15,000 | 0.000026 |     1.5069 |     4.0203 |     34.1550 |     0.3521 |     2829.99 |
| 20,000 | 0.000026 |     1.4650 |     4.1353 |     33.2052 |     0.3304 |     2658.38 |
| 25,000 | 0.000026 |     1.4373 |     4.2151 |     32.5766 |     0.3157 |     2551.04 |
| 30,000 | 0.000026 |     1.4173 |     4.2744 |     32.1241 |     0.3051 |     2470.35 |
| 35,000 | 0.000026 |     1.4019 |     4.3213 |     31.7758 |     0.2967 |     2413.20 |
| 40,000 | 0.000026 |     1.3899 |     4.3585 |     31.5043 |     0.2902 |     2372.07 |
| 45,000 | 0.000026 |     1.3803 |     4.3891 |     31.2847 |     0.2848 |     2336.37 |
| 50,000 | 0.000026 | **1.3721** | **4.4151** | **31.1008** | **0.2803** | **2308.36** |

---

## Table 3. Best BPE Comparison (50k Vocabulary)

| Metric      |       Bangla |        ASCII | Better     |
| ----------- | -----------: | -----------: | :--------- |
| OOV         | **0.000026** | **0.000026** | Tie        |
| Fertility   |   **1.3454** |       1.3721 | **Bangla** |
| CPT         |       3.7880 |   **4.4151** | ASCII*     |
| Avg. Tokens |  **30.4942** |      31.1008 | **Bangla** |
| WFR         |   **0.2698** |       0.2803 | **Bangla** |
| Variance    |  **2224.00** |      2308.36 | **Bangla** |

Higher CPT for ASCII reflects the longer Latin-character transliterations and does not imply better tokenization quality.


In [55]:
import csv

with open(
    "literature_wp_bn_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

    for r in wp_bn_results:

        writer.writerow([
            "Bengali",
            "WordPiece",
            r["vocab"],
            r["oov"],
            r["fertility"],
            r["cpt"],
            r["compression"],
            r["avg_tokens"],
            r["wfr"],
            r["variance"]
        ])

print("Saved.")

Saved.


In [56]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_lit_bn_5000.json"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)
    print(tok.encode(w).tokens)


 বাংলাদেশ
['বাংলা', '##দেশ']

 কলকাতা
['কলকাতা']

 সাহিত্য
['সাহিত্য']

 কবিতা
['কবিতা']

 উপন্যাস
['উপ', '##ন্যাস']


In [57]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO WordPiece-{vocab_size}")

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_lit_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_lit_iso_{vocab_size}.json"
    )

print("Finished.")


Training ISO WordPiece-5000

Training ISO WordPiece-10000

Training ISO WordPiece-15000

Training ISO WordPiece-20000

Training ISO WordPiece-25000

Training ISO WordPiece-30000

Training ISO WordPiece-35000

Training ISO WordPiece-40000

Training ISO WordPiece-45000

Training ISO WordPiece-50000
Finished.


In [58]:
wp_iso_results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_tokenizer(
        f"wp_lit_iso_{vocab_size}.json",
        "test_lit_iso.txt",
        vocab_size
    )

    print(res)

    wp_iso_results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 1.9379162115499805e-05, 'fertility': 1.8368503576336281, 'cpt': 3.2981029758313807, 'compression': 3.2981029758313807, 'avg_tokens': np.float64(41.63371001876772), 'wfr': 0.48362636975441314, 'variance': np.float64(4421.620773150267)}
{'vocab': 10000, 'oov': 1.9379162115499805e-05, 'fertility': 1.6375321517916916, 'cpt': 3.699543623641199, 'compression': 3.699543623641199, 'avg_tokens': np.float64(37.11600047917582), 'wfr': 0.40361333286353546, 'variance': np.float64(3413.122070782574)}
{'vocab': 15000, 'oov': 1.9379162115499805e-05, 'fertility': 1.552406539586343, 'cpt': 3.9024066674459275, 'compression': 3.9024066674459275, 'avg_tokens': np.float64(35.186559118316495), 'wfr': 0.36694443465698884, 'variance': np.float64(3030.5164076108786)}
{'vocab': 20000, 'oov': 1.9379162115499805e-05, 'fertility': 1.501670131425954, 'cpt': 4.034255928707686, 'compression': 4.034255928707686, 'avg_tokens': np.float64(34.036577087409654), 'wfr': 0.3433229977802051, 'variance': 

In [59]:
import csv

with open(
    "literature_wp_iso_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "script",
        "tokenizer",
        "vocab",
        "oov",
        "fertility",
        "cpt",
        "compression",
        "avg_tokens",
        "wfr",
        "variance"
    ])

    for r in wp_iso_results:

        writer.writerow([
            "ISO",
            "WordPiece",
            r["vocab"],
            r["oov"],
            r["fertility"],
            r["cpt"],
            r["compression"],
            r["avg_tokens"],
            r["wfr"],
            r["variance"]
        ])

print("Saved.")

Saved.


In [60]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(
    "wp_lit_iso_5000.json"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)
    print(tok.encode(w).tokens)


 bāṃlā
['b', '##ā', '##ṃ', '##l', '##ā']

 kalakātā
['kala', '##k', '##ā', '##t', '##ā']

 sāhitya
['s', '##ā', '##hi', '##tya']

 kabitā
['kabi', '##t', '##ā']

 upanyāsa
['upan', '##y', '##ā', '##sa']


In [61]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining Bengali Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_lit_bn.txt",
        model_prefix=f"uni_lit_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")


Training Bengali Unigram-5000

Training Bengali Unigram-10000

Training Bengali Unigram-15000

Training Bengali Unigram-20000

Training Bengali Unigram-25000

Training Bengali Unigram-30000

Training Bengali Unigram-35000

Training Bengali Unigram-40000

Training Bengali Unigram-45000

Training Bengali Unigram-50000
Finished


In [62]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_file,
    test_file,
    vocab_size
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_file)

    total_words = 0
    total_tokens = 0
    total_chars = 0

    fragmented = 0
    oov = 0

    token_counts = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            words = line.strip().split()

            total_words += len(words)

            sent_tokens = 0

            for word in words:

                pieces = sp.encode(
                    word,
                    out_type=str
                )

                total_tokens += len(pieces)

                sent_tokens += len(pieces)

                total_chars += len(word)

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

            token_counts.append(
                sent_tokens
            )

    fertility = (
        total_tokens /
        total_words
    )

    cpt = (
        total_chars /
        total_tokens
    )

    compression = cpt

    avg_tokens = np.mean(
        token_counts
    )

    wfr = (
        fragmented /
        total_words
    )

    variance = np.var(
        token_counts
    )

    return {
        "vocab": vocab_size,
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": cpt,
        "compression": compression,
        "avg_tokens": avg_tokens,
        "wfr": wfr,
        "variance": variance
    }

In [63]:
uni_bn_results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_unigram(
        f"uni_lit_bn_{vocab_size}.model",
        "test_lit_bn.txt",
        vocab_size
    )

    print(res)

    uni_bn_results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 0.0, 'fertility': 1.8054423638307955, 'cpt': 2.822723693339331, 'compression': 2.822723693339331, 'avg_tokens': np.float64(40.92189434173222), 'wfr': 0.4570567332780147, 'variance': np.float64(4228.905212048574)}
{'vocab': 10000, 'oov': 0.0, 'fertility': 1.6173855442275744, 'cpt': 3.150927715121637, 'compression': 3.150927715121637, 'avg_tokens': np.float64(36.65942578764525), 'wfr': 0.3899802861416332, 'variance': np.float64(3325.4924826315864)}
{'vocab': 15000, 'oov': 0.0, 'fertility': 1.5330898610164176, 'cpt': 3.3241788801375294, 'compression': 3.3241788801375294, 'avg_tokens': np.float64(34.74879207762648), 'wfr': 0.3561408052203847, 'variance': np.float64(2956.439350355802)}
{'vocab': 20000, 'oov': 0.0, 'fertility': 1.4851458984075643, 'cpt': 3.431491103202847, 'compression': 3.431491103202847, 'avg_tokens': np.float64(33.66210118596015), 'wfr': 0.3359548008266079, 'variance': np.float64(2760.284418809072)}
{'vocab': 25000, 'oov': 0.0, 'fertility': 1.453584

In [64]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load(
    "uni_lit_bn_5000.model"
)

words = [
    "বাংলাদেশ",
    "কলকাতা",
    "সাহিত্য",
    "কবিতা",
    "উপন্যাস"
]

for w in words:

    print("\n", w)

    print(
        sp.encode(
            w,
            out_type=str
        )
    )


 বাংলাদেশ
['▁বাংলাদেশ']

 কলকাতা
['▁কলকাতা']

 সাহিত্য
['▁সাহিত্য']

 কবিতা
['▁কবিতা']

 উপন্যাস
['▁উপন্যাস']


In [65]:
import sentencepiece as spm

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    print(f"\nTraining ISO Unigram-{vocab_size}")

    spm.SentencePieceTrainer.train(
        input="train_lit_iso.txt",
        model_prefix=f"uni_lit_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=-1,
        minloglevel=1
    )

print("Finished")


Training ISO Unigram-5000

Training ISO Unigram-10000

Training ISO Unigram-15000

Training ISO Unigram-20000

Training ISO Unigram-25000

Training ISO Unigram-30000

Training ISO Unigram-35000

Training ISO Unigram-40000

Training ISO Unigram-45000

Training ISO Unigram-50000
Finished


In [66]:
uni_iso_results = []

for vocab_size in [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]:

    res = evaluate_unigram(
        f"uni_lit_iso_{vocab_size}.model",
        "test_lit_iso.txt",
        vocab_size
    )

    print(res)

    uni_iso_results.append(res)

print("\nFinished.")

{'vocab': 5000, 'oov': 0.0, 'fertility': 1.9042916035375779, 'cpt': 3.1812993448137408, 'compression': 3.1812993448137408, 'avg_tokens': np.float64(43.16232080820988), 'wfr': 0.47173637292554876, 'variance': np.float64(4727.805581037201)}
{'vocab': 10000, 'oov': 0.0, 'fertility': 1.7239455974067157, 'cpt': 3.5141025562416073, 'compression': 3.5141025562416073, 'avg_tokens': np.float64(39.07463163359022), 'wfr': 0.4113297628695254, 'variance': np.float64(3761.2999047029834)}
{'vocab': 15000, 'oov': 0.0, 'fertility': 1.6436418730841056, 'cpt': 3.6857917347293143, 'compression': 3.6857917347293143, 'avg_tokens': np.float64(37.25448229046041), 'wfr': 0.3823297276346852, 'variance': np.float64(3392.211443691366)}
{'vocab': 20000, 'oov': 0.0, 'fertility': 1.5965698883055566, 'cpt': 3.7944606590932914, 'compression': 3.7944606590932914, 'avg_tokens': np.float64(36.187557401269814), 'wfr': 0.36485148514851484, 'variance': np.float64(3188.538675194115)}
{'vocab': 25000, 'oov': 0.0, 'fertility':

In [67]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()

sp.load(
    "uni_lit_iso_5000.model"
)

words = [
    "bāṃlā",
    "kalakātā",
    "sāhitya",
    "kabitā",
    "upanyāsa"
]

for w in words:

    print("\n", w)

    print(
        sp.encode(
            w,
            out_type=str
        )
    )


 bāṃlā
['▁b', 'ā', 'ṃ', 'l', 'ā']

 kalakātā
['▁kala', 'k', 'ā', 't', 'ā']

 sāhitya
['▁s', 'ā', 'hi', 'tya']

 kabitā
['▁kabi', 't', 'ā']

 upanyāsa
['▁upa', 'n', 'y', 'ā', 'sa']


Morphological Analysis

In [68]:
from collections import Counter

def hapax_ratio(file_path):

    counter = Counter()

    with open(file_path, encoding="utf-8") as f:

        for line in f:
            counter.update(line.split())

    total_vocab = len(counter)

    hapax = sum(
        1
        for w, c in counter.items()
        if c == 1
    )

    return {
        "vocab": total_vocab,
        "hapax": hapax,
        "ratio": hapax / total_vocab
    }



print("\nLiterature:")
print(hapax_ratio("train_lit_bn.txt"))


Literature:
{'vocab': 486051, 'hapax': 326949, 'ratio': 0.6726639797058334}


In [69]:
from collections import Counter

def vocab_growth(
    file_path,
    step=10000
):

    vocab = set()

    docs = 0

    with open(file_path, encoding="utf-8") as f:

        for line in f:

            docs += 1

            vocab.update(
                line.split()
            )

            if docs % step == 0:

                print(
                    docs,
                    len(vocab)
                )



print("\nLITERATURE")
vocab_growth(
    "train_lit_bn.txt"
)


LITERATURE
10000 33245
20000 76935
30000 125785
40000 148598
50000 171104
60000 182250
70000 208284
80000 223828
90000 245327
100000 264396
110000 278150
120000 294873
130000 322904
140000 356184
150000 371464
160000 390865
170000 405508
180000 428688
190000 443912
200000 456357
210000 471281
220000 481509


In [70]:
from collections import Counter

freq = Counter()

with open(
    "train_lit_bn.txt",
    encoding="utf-8"
) as f:

    for line in f:

        freq.update(
            line.split()
        )

print(
    len(freq)
)

486051


In [71]:
import random

random.seed(42)

items = sorted(
    freq.items(),
    key=lambda x: x[1],
    reverse=True
)

frequent = [
    w for w,c in items[100:5000]
]

medium = [
    w for w,c in items[5000:50000]
]

rare = [
    w for w,c in items[-100000:]
]

selected = (
    random.sample(
        frequent,
        40
    )
    +
    random.sample(
        medium,
        30
    )
    +
    random.sample(
        rare,
        30
    )
)

print(
    len(selected)
)

with open(
    "morph_words_100.txt",
    "w",
    encoding="utf-8"
) as f:

    for w in selected:
        f.write(w + "\n")

100


In [72]:
from tokenizers import Tokenizer
import sentencepiece as spm

bpe = Tokenizer.from_file(
    "bpe_lit_bn_50000.json"
)

wp = Tokenizer.from_file(
    "wp_lit_bn_50000.json"
)

sp = spm.SentencePieceProcessor()
sp.load(
    "uni_lit_bn_50000.model"
)

with open(
    "morph_words_100.txt",
    encoding="utf-8"
) as f:

    words = [
        x.strip()
        for x in f
    ]

for w in words:

    print("\nWORD:", w)

    print(
        "BPE:",
        bpe.encode(w).tokens
    )

    print(
        "WP :",
        wp.encode(w).tokens
    )

    print(
        "UNI:",
        sp.encode(
            w,
            out_type=str
        )
    )


WORD: বাহির
BPE: ['বাহির']
WP : ['বাহির']
UNI: ['▁বাহির']

WORD: রাজা
BPE: ['রাজা']
WP : ['রাজা']
UNI: ['▁রাজা']

WORD: নীরবে
BPE: ['নীরবে']
WP : ['নীরবে']
UNI: ['▁নীরবে']

WORD: ভিজে
BPE: ['ভিজে']
WP : ['ভিজে']
UNI: ['▁ভিজে']

WORD: করিলে
BPE: ['করিলে']
WP : ['করিলে']
UNI: ['▁করিলে']

WORD: করেছি
BPE: ['করেছি']
WP : ['করেছি']
UNI: ['▁করেছি']

WORD: ১.
BPE: ['১', '.']
WP : ['১', '.']
UNI: ['▁১', '.']

WORD: বসবার
BPE: ['বসবার']
WP : ['বসবার']
UNI: ['▁বসবার']

WORD: কর।
BPE: ['কর', '।']
WP : ['কর', '।']
UNI: ['▁কর', '।']

WORD: ট্রেনের
BPE: ['ট্রেনের']
WP : ['ট্রেনের']
UNI: ['▁ট্রেনের']

WORD: কেন।
BPE: ['কেন', '।']
WP : ['কেন', '।']
UNI: ['▁কেন', '।']

WORD: অবশ্য
BPE: ['অবশ্য']
WP : ['অবশ্য']
UNI: ['▁অবশ্য']

WORD: খারাপ
BPE: ['খারাপ']
WP : ['খারাপ']
UNI: ['▁খারাপ']

WORD: জন্য।
BPE: ['জন্য', '।']
WP : ['জন্য', '।']
UNI: ['▁জন্য', '।']

WORD: ফের
BPE: ['ফের']
WP : ['ফের']
UNI: ['▁ফের']

WORD: স্কুলের
BPE: ['স্কুলের']
WP : ['স্কুলের']
UNI: ['▁স্কুলের']

WORD: সেজে
BPE: ['সেজে']
WP : [

In [73]:
from tokenizers import Tokenizer

bpe = Tokenizer.from_file(
    "bpe_lit_bn_50000.json"
)

wp = Tokenizer.from_file(
    "wp_lit_bn_50000.json"
)

import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load(
    "uni_lit_bn_50000.model"
)

def avg_pieces(
    words,
    tokenizer,
    mode
):

    total = 0

    for w in words:

        if mode == "bpe":

            total += len(
                tokenizer.encode(w).tokens
            )

        elif mode == "wp":

            total += len(
                tokenizer.encode(w).tokens
            )

        else:

            total += len(
                tokenizer.encode(
                    w,
                    out_type=str
                )
            )

    return total / len(words)

In [74]:
print("Frequent")

print(
    "BPE",
    avg_pieces(
        frequent,
        bpe,
        "bpe"
    )
)

print(
    "WP",
    avg_pieces(
        frequent,
        wp,
        "wp"
    )
)

print(
    "UNI",
    avg_pieces(
        frequent,
        sp,
        "uni"
    )
)

print("\nMedium")

print(
    "BPE",
    avg_pieces(
        medium,
        bpe,
        "bpe"
    )
)

print(
    "WP",
    avg_pieces(
        medium,
        wp,
        "wp"
    )
)

print(
    "UNI",
    avg_pieces(
        medium,
        sp,
        "uni"
    )
)

print("\nRare")

print(
    "BPE",
    avg_pieces(
        rare,
        bpe,
        "bpe"
    )
)

print(
    "WP",
    avg_pieces(
        rare,
        wp,
        "wp"
    )
)

print(
    "UNI",
    avg_pieces(
        rare,
        sp,
        "uni"
    )
)

Frequent
BPE 1.1895918367346938
WP 1.1895918367346938
UNI 1.2034693877551021

Medium
BPE 1.3813555555555554
WP 1.4561111111111111
UNI 1.5249333333333333

Rare
BPE 2.81226
WP 2.91358
UNI 3.01755
